# 🧪 Micro‑Lab: Sentiment Analysis (Azure AI Language)

Welcome! In ~15 minutes you'll build a tiny but complete Sentiment Analysis app using the Azure AI Language SDK for Python. You can run in Live mode (real Azure calls) or Mock mode (no keys required).

What you'll learn:
- Create a `TextAnalyticsClient` and call `analyze_sentiment`
- Enable Opinion Mining (aspect-based sentiment)
- Handle errors, latency timing, and basic batching
- Map outputs to exam concepts (AI‑102)

Prereqs (Live mode only):
- Azure subscription + Azure AI Language resource
- Set env vars: `AZURE_LANGUAGE_ENDPOINT`, `AZURE_LANGUAGE_KEY`

Tip: If you're not ready with keys, start in Mock mode and switch to Live later.

🔗 Back to NLP README: [05_Implement_Natural_Language_Processing_Solutions/README.md](../README.md)

## 1) Install/Import
If you're running in a local environment, ensure the SDK is installed. In hosted environments, it may already be available.

- First run the install cell to add dependencies
- Then run the import cell
- If you see an "externally-managed-environment" error, run the "Create virtual environment" cell below, switch your Jupyter kernel to "Python (.venv) AI-102 NLP", and re-run the install cell.

In [1]:
# Install required packages (run once)
%pip install --quiet azure-ai-textanalytics azure-identity

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break-system-packages = true' to your pip.conf file. The latter
    will permanently disable this error.
    
    If you disable this error, we STRONGLY recommend that you additionally
    pass the '--user' flag to pip, or set 

In [ ]:
# Create a dedicated virtual environment for this lab (run once)
# After this completes, switch the Jupyter kernel to: Python (.venv) AI-102 NLP
import os, sys, subprocess, json
venv_dir = os.path.join(os.getcwd(), ".venv-ai102-nlp")
python_exe = sys.executable

if not os.path.exists(venv_dir):
    print(f"Creating venv at: {venv_dir}")
    subprocess.run([python_exe, "-m", "venv", venv_dir], check=True)
else:
    print(f"Venv already exists at: {venv_dir}")

# Upgrade pip inside the venv
pip_path = os.path.join(venv_dir, "bin", "pip") if os.name != 'nt' else os.path.join(venv_dir, "Scripts", "pip.exe")
python_path = os.path.join(venv_dir, "bin", "python") if os.name != 'nt' else os.path.join(venv_dir, "Scripts", "python.exe")
subprocess.run([python_path, "-m", "pip", "install", "--upgrade", "pip"], check=True)

# (Optional) Install ipykernel and register a named kernel for easy switching
subprocess.run([python_path, "-m", "pip", "install", "--quiet", "ipykernel"], check=True)
subprocess.run([python_path, "-m", "ipykernel", "install", "--user", "--name", "ai102-nlp", "--display-name", "Python (.venv) AI-102 NLP"], check=True)

print("Virtual environment ready. In the Jupyter UI, change Kernel -> ai102-nlp, then re-run the install cell.")

In [ ]:
# Import libraries (run after the install cell if needed)
import os, time, json
from dataclasses import dataclass
from typing import List, Dict, Any

from azure.core.credentials import AzureKeyCredential
from azure.ai.textanalytics import TextAnalyticsClient
try:
    from azure.identity import DefaultAzureCredential  # optional (managed identity)
except Exception:
    DefaultAzureCredential = None

print('Imports ready')

## 2) Choose Mode: Live or Mock
- Live: real API calls (requires endpoint/key env vars).
- Mock: returns deterministic sample responses (great for practice without keys).

In [ ]:
LIVE_MODE = os.getenv('LANGUAGE_LAB_LIVE', 'false').lower() in ['1','true','yes']
print('Live mode:', LIVE_MODE)

AZURE_LANGUAGE_ENDPOINT = os.getenv('AZURE_LANGUAGE_ENDPOINT')
AZURE_LANGUAGE_KEY = os.getenv('AZURE_LANGUAGE_KEY')
if LIVE_MODE:
    if not AZURE_LANGUAGE_ENDPOINT or not AZURE_LANGUAGE_KEY:
        raise RuntimeError('Live mode requires AZURE_LANGUAGE_ENDPOINT and AZURE_LANGUAGE_KEY env vars.')
else:
    print('Running in Mock mode — you can still complete the lab.')

## 3) Create Client (Live) or Mock Helpers
We'll create a `TextAnalyticsClient` for Live mode, or simple mock functions for offline practice.

In [ ]:
def create_client():
    if not LIVE_MODE:
        return None
    return TextAnalyticsClient(endpoint=AZURE_LANGUAGE_ENDPOINT, credential=AzureKeyCredential(AZURE_LANGUAGE_KEY))

def mock_analyze_sentiment(documents: List[str], show_opinion_mining: bool=False):
    # Minimal deterministic mock to mirror SDK shape for the lab
    class Doc:
        def __init__(self, text):
            self.is_error = False
            self.sentiment = 'positive' if 'love' in text.lower() or 'great' in text.lower() else ('negative' if 'bad' in text.lower() or 'waste' in text.lower() else 'neutral')
            self.confidence_scores = type('Scores', (), {'positive':0.9 if self.sentiment=='positive' else 0.05, 'neutral':0.9 if self.sentiment=='neutral' else 0.05, 'negative':0.9 if self.sentiment=='negative' else 0.05})
            self.sentences = [type('Sent', (), {'text':text, 'sentiment':self.sentiment, 'confidence_scores':self.confidence_scores, 'mined_opinions': []})]
    return [Doc(t) for t in documents]

client = create_client()
client

## 4) Quickstart: Single Document
Try a quick analysis. Edit the text and re-run to see how results change.

In [ ]:
text = "I love the speed of this product, but the battery life is bad."
documents = [text]

start = time.time()
if LIVE_MODE:
    result = client.analyze_sentiment(documents, show_opinion_mining=False)
else:
    result = mock_analyze_sentiment(documents)
elapsed_ms = (time.time()-start)*1000

doc = result[0]
print({'sentiment': doc.sentiment, 'scores': {'pos': doc.confidence_scores.positive, 'neu': doc.confidence_scores.neutral, 'neg': doc.confidence_scores.negative}, 'latency_ms': round(elapsed_ms,1)})

## 5) Batch + Opinion Mining
Batch multiple texts and enable opinion mining to see aspect-level insights (Live mode).

In [ ]:
batch = [
    "The camera quality is great, but the price is too high.",
    "Support was helpful and friendly.",
    "This was a waste of my time."
]

if LIVE_MODE:
    results = client.analyze_sentiment(batch, show_opinion_mining=True)
else:
    results = mock_analyze_sentiment(batch)

parsed = []
for i, r in enumerate(results):
    if r.is_error:
        parsed.append({'error': True})
        continue
    item = {
        'text': batch[i],
        'doc_sentiment': r.sentiment,
        'scores': {'pos': r.confidence_scores.positive, 'neu': r.confidence_scores.neutral, 'neg': r.confidence_scores.negative},
        'sentences': []
    }
    for s in getattr(r, 'sentences', []):
        mined = []
        for mo in getattr(s, 'mined_opinions', []):
            target = getattr(mo, 'target', None)
            assessments = []
            for a in getattr(mo, 'assessments', []):
                assessments.append({'text': a.text, 'sentiment': a.sentiment})
            mined.append({'target': getattr(target, 'text', None), 'sentiment': getattr(target, 'sentiment', None), 'assessments': assessments})
        item['sentences'].append({'text': s.text, 'sentiment': s.sentiment, 'mined_opinions': mined})
    parsed.append(item)

print(json.dumps(parsed, indent=2))

## 6) Error Handling Patterns
Common errors include missing keys, wrong endpoint region, and payload size. In batch results, check `is_error` per item.

In [ ]:
def safe_analyze(documents: List[str], **kwargs):
    try:
        if LIVE_MODE:
            return client.analyze_sentiment(documents, **kwargs)
        return mock_analyze_sentiment(documents, **kwargs)
    except Exception as e:
        print('Caught error:', type(e).__name__, str(e)[:200])
        return []

bad_docs = ['']  # empty doc to simulate a per-item error in Live mode
res = safe_analyze(bad_docs, show_opinion_mining=True)
if res:
    for r in res:
        print('is_error:', getattr(r, 'is_error', None), getattr(r, 'error', None))
else:
    print('Handled safely.')

## 7) Challenge (5–10 min)
- Swap in your own sentences and see how scores change.
- Try setting `language='es'` on `analyze_sentiment` and input Spanish text — what changes?
- Compare results with and without `show_opinion_mining=True` for product review text.

## 8) Mini‑Quiz (3 Qs)
1) What extra insight does Opinion Mining provide beyond overall sentiment?
2) Name two common error causes when calling the API.
3) How do you disable service logs in the SDK call? (hint: parameter name)

Answers (hover/select to reveal):
<details><summary>Show</summary>Aspect/target and assessments per aspect; bad key/endpoint or payload/size issues; use `disable_service_logs=True`.</details>

## 9) Reference
- SDK: `azure-ai-textanalytics` → `TextAnalyticsClient.analyze_sentiment`
- Docs: https://learn.microsoft.com/azure/ai-services/language-service/sentiment-opinion-mining/overview
- Python API: https://learn.microsoft.com/python/api/azure-ai-textanalytics/azure.ai.textanalytics.textanalyticsclient
- Data limits: https://aka.ms/azsdk/textanalytics/data-limits

Next: Try the Key Phrase Extraction lab.